In [17]:
%pip install pyspark scikit-learn

In [18]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.sql.functions import col


spark = SparkSession.builder.appName("SentimentAnalysis").getOrCreate()

data_path = "sentiments.csv"
df = spark.read.csv(data_path, header=True, inferSchema = True)
# Convert -1/1 labels to 0/1: Normalize sentiment labels
df = df.withColumn("label", (col("sentiment").cast("integer") + 1) / 2)
# Drop rows with null sentiment values before processing
initial_row_count = df.count()
df = df.dropna(subset=["sentiment"])

tokenizer = Tokenizer(inputCol="text", outputCol="words")

stopwordsRemover = StopWordsRemover(inputCol="words", outputCol="filtered_words")

hashingTF = HashingTF(inputCol="filtered_words", outputCol="raw_features",numFeatures=10000)


idf = IDF(inputCol="raw_features", outputCol="features")

In [19]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(maxIter=10,regParam=0.001 ,featuresCol="features", labelCol="label")

In [20]:
from pyspark.ml import Pipeline
pipeline = Pipeline(stages=[tokenizer, stopwordsRemover, hashingTF, idf, lr])


In [21]:
#Train-Test split on Spark dataframe
train_ratio = 0.8
test_ratio = 0.2
weights = [train_ratio, test_ratio]
df = df[['text', 'label']]
seed = 42
train_df, test_df = df.randomSplit(weights, seed=seed)


In [22]:
model = pipeline.fit(train_df)

In [23]:
y_pred = model.transform(test_df)

In [29]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

y_true = test_df.select("label").collect()

evaluator = MulticlassClassificationEvaluator()

#Đánh giá bằng 2 chỉ số accuracy và f1-score:
accuracy = evaluator.evaluate(y_pred,{evaluator.metricName:"accuracy"})
f1 = evaluator.evaluate(y_pred, {evaluator.metricName:"f1"})
print("Accuracy:", accuracy)
print("f1", f1)

Accuracy: 0.7294860234445446
f1 0.7266221530017497


Nhận xét:
- Mô hình có độ chính xác tốt lên tới trên 0.70 trên các chỉ số

Task 4: Improving Model Performance

Phương pháp lựa chọn: thay TF-IDF trong pipeline bằng Word2Vec

In [31]:
from pyspark.ml.feature import Word2Vec

w2v = Word2Vec(vectorSize=100, inputCol="filtered_words", outputCol="features")

pipeline = Pipeline(stages = [tokenizer, stopwordsRemover, w2v, lr])

In [32]:
model = pipeline.fit(train_df)
y_pred = model.transform(test_df)

accuracy = evaluator.evaluate(y_pred,{evaluator.metricName:"accuracy"})
f1 = evaluator.evaluate(y_pred, {evaluator.metricName:"f1"})
print("Accuracy:", accuracy)
print("f1", f1)

Accuracy: 0.6447249774571686
f1 0.5787382551010488


Nhận xét:
- Mô hình Word2Vec khi được train trên bộ dữ liệu có kết quả kém hơn so với sử dụng hashingTF + IDF
  - Điều trên có thể do Word2Vec phụ thuộc nhiều vào kích thước bộ dữ liệu để đạt được kết quả tốt.